# `gf-persona-data` 정제 연습

`korean-role-playing` 안의 `gf-persona-data`만 불러옵니다. 다른 데이터셋은 사용하지 않습니다.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
GF_PATH = ROOT / 'data' / 'raw' / 'huggingface-krew' / 'korean-role-playing' / 'gf-persona-data' / 'train-00000-of-00001.parquet'

if not GF_PATH.exists():
    raise FileNotFoundError(f'gf 데이터가 없습니다: {GF_PATH}')

print(f'불러오는 파일: {GF_PATH}')
print(f'파일 크기: {GF_PATH.stat().st_size / 1024**2:.2f} MB')

## 1. 원본 불러오기와 구조 확인

In [ ]:
df = pd.read_parquet(GF_PATH)
print(f'행 수: {len(df):,}')
print(f'컬럼: {list(df.columns)}')
display(df.head(3))
display(df.dtypes.to_frame('dtype'))

## 2. 원본 대화 한 건 자세히 보기

`text`는 `role`과 `content`를 가진 메시지 목록입니다.

In [ ]:
def show_conversation(row):
    print(f'topic: {row.get("topic")}')
    for i, message in enumerate(row['text']):
        print(f'[{i}] {message.get("role")}: {message.get("content")}')

show_conversation(df.iloc[0])

## 3. 정제 전 품질 점검

In [ ]:
def message_texts(messages):
    if not isinstance(messages, list):
        return []
    return [m.get('content', '') if isinstance(m, dict) else '' for m in messages]

def has_valid_messages(messages):
    return isinstance(messages, list) and len(messages) > 0 and all(
        isinstance(m, dict) and m.get('role') in {'user', 'assistant'} and str(m.get('content', '')).strip()
        for m in messages
    )

df['message_count'] = df['text'].map(lambda x: len(x) if isinstance(x, list) else 0)
df['char_count'] = df['text'].map(lambda x: sum(len(t) for t in message_texts(x)))
df['valid_messages'] = df['text'].map(has_valid_messages)

print('결측치')
display(df.isna().sum().to_frame('null_count'))
print('topic 분포')
display(df['topic'].value_counts(dropna=False))
print('대화 턴 수 / 문자 수')
display(df[['message_count', 'char_count']].describe())
print(f'정상 형식 행: {df.valid_messages.sum():,} / {len(df):,}')
df['_dedup_key'] = df.apply(lambda row: json.dumps(row['text'], ensure_ascii=False, sort_keys=True) + '|' + str(row['topic']), axis=1)
print(f'완전 중복 행: {df["_dedup_key"].duplicated().sum():,}')

## 4. 기본 정제 함수

연습용 기본 규칙입니다.

- 메시지 앞뒤 공백 제거
- 연속 공백 정리
- 빈 메시지 제거
- `user`·`assistant` role만 유지
- 완전히 중복된 대화 제거

In [ ]:
def clean_content(content):
    content = str(content).replace('\u00a0', ' ')
    content = re.sub(r'[ \t]+', ' ', content)
    content = re.sub(r'\n{3,}', '\n\n', content)
    return content.strip()

def clean_messages(messages):
    cleaned = []
    for message in messages if isinstance(messages, list) else []:
        if not isinstance(message, dict) or message.get('role') not in {'user', 'assistant'}:
            continue
        content = clean_content(message.get('content', ''))
        if content:
            cleaned.append({'role': message['role'], 'content': content})
    return cleaned

clean_df = df[['text', 'topic']].copy()
clean_df['text'] = clean_df['text'].map(clean_messages)
clean_df = clean_df[clean_df['text'].map(has_valid_messages)].copy()
clean_df['_dedup_key'] = clean_df.apply(lambda row: json.dumps(row['text'], ensure_ascii=False, sort_keys=True) + '|' + str(row['topic']), axis=1)
clean_df = clean_df.drop_duplicates(subset=['_dedup_key']).drop(columns='_dedup_key').reset_index(drop=True)

print(f'정제 전: {len(df):,}행')
print(f'정제 후: {len(clean_df):,}행')
print(f'제거된 행: {len(df) - len(clean_df):,}행')
display(clean_df.head(3))

## 5. 정제 결과 저장

원본은 보존하고, 정제본만 별도 경로에 저장합니다.

In [ ]:
OUTPUT_PATH = ROOT / 'data' / 'cleaned' / 'gf-persona-clean.parquet'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_parquet(OUTPUT_PATH, index=False)
print(f'저장 완료: {OUTPUT_PATH}')